# 04 · RQ1 analysis

Joins the detector scores with our labels (`final` in `labels.csv`) and the technique labels, and builds the RQ1 tables and Figure 1. Input: `scores_rq1.csv`, `labels.csv`, `techniques.csv`. Output: `results_rq1/`.

Metrics use our labels. Each sampled regular prompt has weight 13.1 (13,122 regular prompts / 1,000 sampled), so the results represent the full dataset.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/jailbreak_project"
OUT_DIR = f"{SAVE_DIR}/results_rq1"
os.makedirs(OUT_DIR, exist_ok=True)

DET = {"protectai": "ProtectAI", "pg2_22m": "PG2-22M", "pg2_86m": "PG2-86M", "piguard": "PIGuard"}
CO = ["role_play", "refusal_suppression", "style_injection", "prefix_injection"]
MG = ["special_encoding", "character_transformation", "word_transformation", "prompt_level_transformation"]
TECH = CO + MG + ["other"]
NAME = {"role_play": "Role-play", "refusal_suppression": "Refusal suppression", "style_injection": "Style injection",
        "prefix_injection": "Prefix injection", "special_encoding": "Special encoding",
        "character_transformation": "Character transformation", "word_transformation": "Word transformation",
        "prompt_level_transformation": "Prompt-level transformation", "other": "Other"}

scores = pd.read_csv(f"{SAVE_DIR}/scores_rq1.csv")
labels = pd.read_csv(f"{SAVE_DIR}/labels.csv", keep_default_na=False)
tech = pd.read_csv(f"{SAVE_DIR}/techniques.csv")

x = labels.merge(scores, on="uid", how="left").merge(tech[["uid"] + TECH], on="uid", how="left")
x[TECH] = x[TECH].fillna(0).astype(int)
x["any_mg"] = x[MG].max(axis=1)
assert len(x) == 2356 and (x.orig == x.label).all()

y_o = x.orig.values                          # original tag
y_n = (x.final == "yes").values.astype(int)  # our label
w = np.where(y_o == 1, 1.0, (scores.label == 0).sum() / (y_o == 0).sum())
T = 0.5                                      # detection threshold
print(f"{y_n.sum()} jailbreaks, {(y_n == 0).sum()} not")
print(f"Tagged jailbreak, not a jailbreak under our labels: {100 * (1 - y_n[y_o == 1].mean()):.1f}%")
print(f"Tagged regular, a jailbreak under our labels: {100 * y_n[y_o == 0].mean():.1f}%")
print("Final jailbreak label from:", labels.source.value_counts().to_dict())
print("Technique label from:", tech.source.value_counts().to_dict())


def wrate(flag, mask):
    """Weighted share of flagged prompts within mask."""
    return (w[mask] * flag[mask]).sum() / w[mask].sum()

## 1. AUC, recall and FPR

The last column scores the same predictions against the original tags. The ProtectAI − PIGuard interval uses 2,000 bootstrap resamples within each tag group. The last cell sets each other detector's threshold so that it flags the same share of benign prompts as ProtectAI does at 0.5, and reports its recall there.

In [ ]:
rows = []
for d, name in DET.items():
    s = x[d].values
    flag = (s >= T).astype(float)
    rows.append({"Detector": name,
                 "AUC": round(roc_auc_score(y_n, s, sample_weight=w), 3),
                 "Recall (%)": round(100 * wrate(flag, y_n == 1), 1),
                 "FPR (%)": round(100 * wrate(flag, y_n == 0), 1),
                 "AUC (original tags)": round(roc_auc_score(y_o, s), 3)})
t = pd.DataFrame(rows)
print(t.to_string(index=False))
t.to_csv(f"{OUT_DIR}/main_table1_detectors.csv", index=False)

In [ ]:
rng = np.random.default_rng(2026)
J_all, R_all = np.where(y_o == 1)[0], np.where(y_o == 0)[0]
diffs = []
for _ in range(2000):
    i = np.r_[rng.choice(J_all, len(J_all)), rng.choice(R_all, len(R_all))]
    diffs.append(roc_auc_score(y_n[i], x.protectai.values[i], sample_weight=w[i])
                 - roc_auc_score(y_n[i], x.piguard.values[i], sample_weight=w[i]))
print(f"ProtectAI - PIGuard AUC: {np.mean(diffs):+.3f} [{np.percentile(diffs, 2.5):+.3f}, {np.percentile(diffs, 97.5):+.3f}]")

In [ ]:
def recall_at_fpr(s, target):
    """Weighted recall at the lowest threshold whose weighted FPR does not exceed target."""
    for t in np.unique(s[y_n == 0]):                 # benign scores, ascending
        if wrate(s > t, y_n == 0) <= target:
            return wrate(s > t, y_n == 1)


target = wrate(x.protectai.values >= T, y_n == 0)
print(f"Recall at ProtectAI's FPR ({100 * target:.1f}%):",
      ", ".join(f"{name} {100 * recall_at_fpr(x[d].values, target):.1f}%" for d, name in DET.items() if d != "protectai"))

## 2. Technique counts

Counts over the 1,251 jailbreaks. The rubric conventions are in `labeling/rubric_techniques.md`.

In [ ]:
jb = x[y_n == 1]
t = pd.DataFrame({"Technique": [NAME[c] for c in TECH],
                  "Jailbreaks": jb[TECH].sum().values,
                  "%": (100 * jb[TECH].mean()).round(1).values})
print(t.to_string(index=False))
print(f"Techniques per jailbreak: {jb[TECH].sum(axis=1).mean():.1f} on average")
t.to_csv(f"{OUT_DIR}/main_table2_technique_counts.csv", index=False)

## 3. Recall by technique

Weighted recall on jailbreaks with and without each competing-objectives technique (Figure 1). Role-play is in almost every jailbreak, so it is not compared; recall on the jailbreaks whose only competing-objectives technique is role-play is printed instead.

In [ ]:
J = y_n == 1
CMP = ["refusal_suppression", "prefix_injection", "style_injection"]
rows = []
for c in CMP:
    has = x[c].values == 1
    for d, name in DET.items():
        flag = x[d].values >= T
        rows.append({"Technique": NAME[c], "Detector": name, "n with": (J & has).sum(), "n without": (J & ~has).sum(),
                     "Recall with": round(100 * wrate(flag, J & has), 1),
                     "Recall without": round(100 * wrate(flag, J & ~has), 1)})
t = pd.DataFrame(rows)
print(t.to_string(index=False))
t.to_csv(f"{OUT_DIR}/fig1_recall_by_technique.csv", index=False)

only_rp = J & (x.role_play.values == 1) & (x[CMP].sum(axis=1).values == 0)
print(f"Role-play only (n = {only_rp.sum()}):",
      ", ".join(f"{name} {100 * wrate(x[d].values >= T, only_rp):.1f}%" for d, name in DET.items()))

In [ ]:
fpr = {name: 100 * wrate(x[d].values >= T, y_n == 0) for d, name in DET.items()}
fig, axes = plt.subplots(1, len(DET), figsize=(9, 2.4), sharey=True)
for ax, name in zip(axes, DET.values()):
    r = t[t.Detector == name]
    pos = np.arange(len(r))
    ax.hlines(pos, r["Recall without"], r["Recall with"], color="#b8b8b8", linewidth=2, zorder=1)
    ax.scatter(r["Recall with"], pos, s=46, color="#1f4e79", zorder=2, clip_on=False, label="With the technique")
    ax.scatter(r["Recall without"], pos, s=46, color="#e07b39", zorder=2, clip_on=False, label="Without")
    ax.set_title(f"{name} (FPR {fpr[name]:.1f}%)", fontsize=10)
    ax.set_xlim(0, 100)
    ax.set_xticks([0, 50, 100])
    ax.set_ylim(len(r) - 0.5, -0.5)
    ax.grid(axis="x", color="#e6e6e6", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(length=0, labelsize=9)
    ax.spines[["top", "right", "left"]].set_visible(False)
axes[0].set_yticks(np.arange(len(CMP)), [NAME[c] for c in CMP])
fig.supxlabel("Recall (%)", fontsize=9, y=0.06)
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=2, frameon=False, fontsize=9,
           bbox_to_anchor=(0.55, 1.08))
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/fig1_recall_by_technique.png", dpi=200, bbox_inches="tight")
plt.show()

### Checks for the gaps above 30 points

Adjusted gap: from a weighted logistic regression (caught ~ the four competing-objectives techniques + any mismatched-generalization technique + log prompt length), the change in recall if every jailbreak had the technique versus none had it (average marginal effect). Tagged only: recall on jailbreak-tagged prompts only, without weights.

In [ ]:
X = np.c_[x[CO + ["any_mg"]].values[J], np.log(x.words.values[J])].astype(float)
tagged = J & (y_o == 1)
rows = []
for d, name in DET.items():
    flag = x[d].values >= T
    model = LogisticRegression(C=100.0, max_iter=2000).fit(X, flag[J], sample_weight=w[J])
    for c in CMP:
        has, k = x[c].values == 1, CO.index(c)
        X1, X0 = X.copy(), X.copy()
        X1[:, k], X0[:, k] = 1, 0
        ame = np.average(model.predict_proba(X1)[:, 1] - model.predict_proba(X0)[:, 1], weights=w[J])
        rows.append({"Technique": NAME[c], "Detector": name,
                     "Gap": round(100 * (wrate(flag, J & has) - wrate(flag, J & ~has)), 1),
                     "Adjusted gap": round(100 * ame, 1),
                     "Tagged only, with": round(100 * flag[tagged & has].mean(), 1),
                     "Tagged only, without": round(100 * flag[tagged & ~has].mean(), 1)})
t = pd.DataFrame(rows)
print(t[t.Gap > 30].to_string(index=False))

## 4. Prompt length

Prompt length in words (split on whitespace). FPR by length band, and the median length of some groups of jailbreaks.

In [ ]:
band = pd.cut(x.words, [0, 100, 300, 1000, np.inf], right=False, labels=["<100", "100-299", "300-999", "1,000+"]).values
t = pd.DataFrame({"Words": band.categories, "Benign prompts": [((y_n == 0) & (band == b)).sum() for b in band.categories]})
for d, name in DET.items():
    t[f"{name} FPR (%)"] = [round(100 * wrate(x[d].values >= T, (y_n == 0) & (band == b)), 1) for b in band.categories]
print(t.to_string(index=False))

rs = x.refusal_suppression.values == 1
for label, m in [("all jailbreaks", J), ("with refusal suppression", J & rs), ("without refusal suppression", J & ~rs),
                 ("role-play only", only_rp), ("benign prompts", y_n == 0)]:
    print(f"Median words, {label}: {np.median(x.words.values[m]):.0f}")